importing needed libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError as e:
    raise ImportError("Install xgboost first: pip install xgboost --break-system-packages") from e

RANDOM_STATE = 42

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame  # includes target column 'MedHouseVal'
TARGET = "MedHouseVal"

print(df.head())
print(df.describe())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.3333

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df[TARGET], kde=True, bins=40)
plt.title("Target distribution: Median House Value")
plt.tight_layout()
plt.savefig("eda_target_distribution.png")
plt.close()

# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png")
plt.close()

In [ ]:
top_corr_feats = df.corr(numeric_only=True)[TARGET].abs().sort_values(ascending=False).index[1:4]
for feat in top_corr_feats:
    plt.figure(figsize=(5, 4))
    sns.scatterplot(x=df[feat], y=df[TARGET], alpha=0.3)
    plt.title(f"{feat} vs {TARGET}")
    plt.tight_layout()
    plt.savefig(f"eda_scatter_{feat}.png")
    plt.close()

print("\nEDA plots saved: eda_target_distribution.png, eda_correlation_heatmap.png, eda_scatter_*.png")


EDA plots saved: eda_target_distribution.png, eda_correlation_heatmap.png, eda_scatter_*.png


In [ ]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    # Ratio feature: rooms per household
    out["rooms_per_household"] = out["AveRooms"] / out["AveOccup"].replace(0, np.nan)
    # Ratio feature: bedrooms per room
    out["bedrooms_per_room"] = out["AveBedrms"] / out["AveRooms"].replace(0, np.nan)
    # Interaction feature: income x house age (captures "established wealthy area" effect)
    out["income_x_age"] = out["MedInc"] * out["HouseAge"]
    # Log transform of a skewed feature
    out["log_population"] = np.log1p(out["Population"])
    out = out.fillna(out.median(numeric_only=True))
    return out


df_fe = engineer_features(df)
new_feats = ["rooms_per_household", "bedrooms_per_room", "income_x_age", "log_population"]
print("\nNew engineered features:", new_feats)

X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# ---------------------------------------------------------------------------
# 4. MODELS + TUNING
# ---------------------------------------------------------------------------
results = []  # (model_name, MAE, RMSE, R2)
predictions = {}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results.append((name, round(mae, 4), round(rmse, 4), round(r2, 4)))
    print(f"\n=== {name} ===\nMAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    predictions[name] = y_pred


# --- Ridge (tuned with GridSearchCV) ---
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(random_state=RANDOM_STATE)),
])
ridge_grid = {"model__alpha": [0.01, 0.1, 1, 10, 50, 100]}
ridge_search = GridSearchCV(ridge_pipe, ridge_grid, scoring="neg_mean_squared_error", cv=cv, n_jobs=-1)
ridge_search.fit(X_train, y_train)
print("Best Ridge alpha:", ridge_search.best_params_)
ridge_preds = ridge_search.predict(X_test)
report("Ridge (tuned)", y_test, ridge_preds)

# --- RandomForestRegressor (tuned with RandomizedSearchCV) ---
rf_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 8, 12, 20],
    "min_samples_split": [2, 5, 10],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    rf_grid, n_iter=12, scoring="neg_mean_squared_error", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
rf_preds = rf_search.predict(X_test)
report("RandomForestRegressor (tuned)", y_test, rf_preds)

# --- XGBRegressor (tuned with RandomizedSearchCV) ---
xgb_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
}
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, objective="reg:squarederror"),
    xgb_grid, n_iter=12, scoring="neg_mean_squared_error", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_search.fit(X_train, y_train)
print("Best XGB params:", xgb_search.best_params_)
xgb_preds = xgb_search.predict(X_test)
report("XGBRegressor (tuned)", y_test, xgb_preds)

# ---------------------------------------------------------------------------
# 5. PREDICTED VS ACTUAL PLOTS (one per model)
# ---------------------------------------------------------------------------
for name, preds in predictions.items():
    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, preds, alpha=0.3)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"Predicted vs Actual - {name}")
    plt.tight_layout()
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(f"pred_vs_actual_{safe_name}.png")
    plt.close()

# ---------------------------------------------------------------------------
# 6. MODEL COMPARISON TABLE
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
print("\n\n===== MODEL COMPARISON =====")
print(results_df.to_string(index=False))
results_df.to_csv("task1_model_comparison.csv", index=False)

# ---------------------------------------------------------------------------
# 7. JUSTIFICATION (fill in after you see your own numbers)
# ---------------------------------------------------------------------------
# Typically on California Housing: XGBRegressor > RandomForest > Ridge on R2/RMSE,
# because the target has non-linear relationships (e.g. income x location) that
# Ridge's linear form can't capture, while boosting handles them well without
# overfitting as easily as a single deep tree. Ridge remains the most
# interpretable and fastest to train - useful if you need to explain
# coefficients to a non-technical stakeholder.
print("\nBest model by R2:", results_df.loc[results_df['R2'].idxmax(), 'Model'])


New engineered features: ['rooms_per_household', 'bedrooms_per_room', 'income_x_age', 'log_population']
Best Ridge alpha: {'model__alpha': 50}

=== Ridge (tuned) ===
MAE=0.4843  RMSE=0.6758  R2=0.6514


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best RF params: {'n_estimators': 400, 'min_samples_split': 5, 'max_depth': 20}

=== RandomForestRegressor (tuned) ===
MAE=0.3294  RMSE=0.5055  R2=0.8050
Best XGB params: {'subsample': 0.9, 'n_estimators': 600, 'max_depth': 5, 'learning_rate': 0.1}

=== XGBRegressor (tuned) ===
MAE=0.2880  RMSE=0.4414  R2=0.8513


===== MODEL COMPARISON =====
                        Model    MAE   RMSE     R2
                Ridge (tuned) 0.4843 0.6758 0.6514
RandomForestRegressor (tuned) 0.3294 0.5055 0.8050
         XGBRegressor (tuned) 0.2880 0.4414 0.8513

Best model by R2: XGBRegressor (tuned)
